# Requirements


In [ ]:
import asyncio
from pathlib import Path
from typing import Dict, Any, Optional, List
from pydantic import BaseModel
from playwright.async_api import Page, Locator
from commons.consts.environment import GbsConsts, PathConsts
from commons.schemas.avw_art import AvwArt, ArtDimension, ArtType
from commons.schemas.ui_entity_mapping import ScrapingListingItem, UIInputIdElement
from commons.utils.playwright import start_page, display_page
from commons.utils.playwright_with_ui_entity_mapping import CustomFill
from commons.utils.dict import MatchType, subdict, is_match_dict
from commons.utils.math import format_number_ptbr
from commons.helpers.ui_entity_processor import (
    UiLoginProcessor,
    UiEntityWithListingProcessor,
    UiEntityWithListingModalProcessor,
)

In [ ]:
!playwright install > /dev/null
!playwright install-deps > /dev/null

# Inputs


# Functions


In [ ]:
def get_mapping_path(file_name: str) -> Path:
    return Path(f"{PathConsts.DATA_PATH}/gbs_store/mapping/{file_name}")

In [ ]:
def find_match(
    items: List[ScrapingListingItem],
    match: Dict[str, Any],
    match_type: Optional[MatchType] = None,
) -> Optional[Dict[str, Any]]:

    for item in items:
        data = item.data
        if is_match_dict(data, match, match_type):
            return data
    return None


def find_field(
    items: List[ScrapingListingItem],
    match: Dict[str, Any],
    field_name: str,
    match_type: Optional[MatchType] = None,
) -> Optional[str]:
    item = find_match(items, match, match_type) or {}
    return item.get(field_name)

In [ ]:
async def fill_input_code(
    form: Locator, input: UIInputIdElement, value: Any, page: Page
):
    if not isinstance(value, str):
        raise ValueError(f"O campo {input.id_} possui valor inválido.")

    await form.locator(f"input[name='{input.name}']").first.click()

    code_editor = page.locator(".modal-content #code .CodeMirror")
    await code_editor.wait_for(state="visible", timeout=10000)

    await code_editor.click()

    await page.keyboard.press("Control+A")
    await page.keyboard.press("Backspace")
    await page.keyboard.type(value)

    await page.locator(".modal-content #salvar-modal-large").first.click()

In [ ]:
async def create_or_update(
    page: Page,
    file_mapping: str,
    data: Dict[str, Any],
    match: Optional[Dict[str, str]] = None,
    match_type: Optional[MatchType] = None,
    checker_field_name: Optional[str] = "id",
    url_params: Optional[Dict[str, str]] = None,
    is_modal: bool = False,
):
    if match is None:
        match = subdict(data, ["name"])

    custom_fill: CustomFill = {"text_code_modal": fill_input_code}

    if is_modal:
        entity_proc = UiEntityWithListingModalProcessor(
            page, get_mapping_path(file_mapping), url_params, custom_fill=custom_fill
        )
    else:
        entity_proc = UiEntityWithListingProcessor(
            page, get_mapping_path(file_mapping), url_params, custom_fill=custom_fill
        )

    field = find_field(
        await entity_proc.list_items(), match, checker_field_name, match_type
    )
    if field is None:
        await entity_proc.create_item(data)
        field = find_field(
            await entity_proc.list_items(), match, checker_field_name, match_type
        )
    else:
        await entity_proc.update_item(data, match, match_type)

    if field is None:
        raise ValueError("Falha ao criar/atualizar item")

    return entity_proc, field

In [ ]:
def dimension_to_text(dimension: ArtDimension):
    refs = []
    dims = []

    if dimension.width:
        refs.append("L")
        dims.append(f"{dimension.width}cm")

    if dimension.height:
        refs.append("A")
        dims.append(f"{dimension.height}cm")

    if dimension.depth:
        refs.append("P")
        dims.append(f"{dimension.depth}cm")

    return " x ".join(dims) + f" ({' x '.join(refs)})"

In [ ]:
def create_synthetic_description(art: AvwArt):
    texts = [
        f"Obra: {art.name}",
        f"Artista: {art.artist_name}",
    ]

    if art.year:
        texts.append(f"Ano da obra: {art.year}")

    texts.append(f"Dimensões totais: {dimension_to_text(art.dimensions[0])}")

    if art.materials_tags and len(art.materials_tags) > 0:
        texts.append(f"Meteriais: {'. '.join(art.materials_tags)}")

    if art.delivery_information:
        texts.append(art.delivery_information)

    if art.stock_information:
        texts.append(art.stock_information)

    if art.extra_description:
        texts.append(art.extra_description)

    return "\n".join(texts)

In [ ]:
class GbsData(BaseModel):
    resource: Dict[str, Any]
    resource_price: Dict[str, Any]
    product: Dict[str, Any]
    product_parts: Dict[str, Any]
    product_part_resources: Dict[str, Any]
    composition: Dict[str, Any]
    composition_library: Dict[str, Any]

In [ ]:
def get_gbs_entity_base_date(art: AvwArt) -> GbsData:

    is_sculpture = art.type == ArtType.SCULPTURE

    resource = {
        "name": f"Rec_{art.name.replace(' ', '_')}",  # Rec_O_Abraço
        "reference": "Escultura" if is_sculpture else "Quadro",  # Escultura ou Quadro
        "resource_type": "INSUMO",  # deixar assim
        # "volume_separator": "",  # deixar vazio
        "is_finish_not_verified": True,  # deixar true
        "unit_measure": "28",  # 28-unidade
    }

    resource_price = {
        "identifier": "preco_1",
        # "external_reference": "",  # deixar em branco
        "break": 0.0,  # sempre 0.0
        "unit_purchase_price": art.price,  # preco do produto
        "equivalence": 1.0,  # sempre 1.0
        "unit_purchase": "28",  # 28-unidade
        "rounding": False,  # deixar false
        "price_group": "29231"
        if is_sculpture
        else "29232",  # 29231-Aviwa_Escultura ou 29232-Aviwa_Quadros
        "finishing": "87015",  # valor 87015 - 'nenhum'
    }

    product = {
        "name": f"{art.name}_{art.artist_name}",  # O Abraço_Elisa Zaterra?  Nome do produto – Colocar nome da peça como no site Aviwa seguido do nome do artista
        "reference": art.code,  # essa referencia vai ser usada na composição - Referência – Colocar código da peça como no site viwa
        "slug": "Escultura" if is_sculpture else "Quadro",  # Escultura ou Quadro
        "product_type": "COMPRADO",  # COMPRADO smpre
        "product_group": "738",  # sempre 738-GrupoAviwa
        # "aplication_group": "",  # para teste usar o 23-GrupoTesteAplicacao
        "unit_measure": "28",  # 28-unidade
    }

    product_parts = {
        "name": f"{'Esc' if is_sculpture else 'Q'}_{art.name}",  # Esc_ ou Q_ou Obj + o nome do produto  Esc_O_Abraço
        "weight": format_number_ptbr(art.dimensions[0].weight)
        if art.dimensions[0].weight
        else "",  # se tiver colcoar
        "is_show_appointment": True,  # sempre true
        "measurements_order": "LxPxA",  # sempre LxPxA
        "unit_weight": "Kg/Un",  # deixar esse
        "length": f"$R= {int(art.dimensions[0].height)}",  # medida em cm
        "depth_width": f"$R= {int(art.dimensions[0].width)}",  # medida em cm
        "height_thickness": f"$R= {int(art.dimensions[0].depth)}"
        if art.dimensions[0].depth
        else "",  # medida em cm
        "is_show_data": True,  # sempre true
        "condition": "1",  # sempre 1
        "part_type": '$R= "C"',  # deixar esse
    }

    product_part_resources = {
        # resource_id # aqui e o recurso id
        "calc_group": "4751",  # 4751-Aviwa
        "quantity": "$R= 1",  # deixar assim
        # "length": "",
        # "depth_width": "",
        # "height_thickness": "",
        # "finishing": "",
        # "configuration": "",
        # "description": "",
        "optimization_type": "NENHUMA",  # deixar  NENHUMA - Sem otimização
        "is_show_data": True,  # deixar assim
    }

    composition = (
        {
            "name": f"{art.name}_{art.artist_name}",  # Colocar nome da Obra_Nome Artista Notro_Karen Haas
            "reference": art.code,  # tem que ser a mesma referencia do cadastro do produto**
            # "version": "",
            # "minimum_version": "",
            # "is_glue_component": "",
            # "is_replacement_component": "",
            # "allows_duplication": "",
            # "is_constructor": "",
            # "with_pagination": "",
            # "with_hole": "",
            "generation": "0",  # 0 - Geração Antiga
            "file_type": "UNICO",  # UNICO - Único
            "details": create_synthetic_description(
                art
            ),  # colocar descricao conforme ppt
            "link": art.url,  # url do site
            # "skp_file": Path(f"{PathConsts.DATA_PATH}/raw_skps/M3D000012.skp"),
            # "image_icon": "",
            # "extra_image_1": "",
            # "extra_image_2": "",
            # "extra_image_3": "",
            # "extra_image_4": "",
            # "extra_image_5": "",
        }
        | {
            ("image_icon" if i == 0 else f"extra_image_{i}"): url
            for i, url in enumerate(art.image_urls)
            # TODO: Deixar as imagens quadradas e deixar branco o que sobra
        }
    )

    # TODO verificar com a Ale
    composition_library = {
        "library": "2583"
        if is_sculpture
        else "2584",  #  2583-Esculturas e Objetos; 2584-Quadros e Telas;
        "level_1": "8648",
        "level_2": "Figurativo",
    }
    # se o library Esculturaqs e objetos
    # level_1 - conjuntos, esculturas, objetos
    # level_2 - Organico, figurativo, figura humanda, abstrato, geometrico

    # se o library Qadros e telas
    # level_1 - conjuntos, impressos, pinturas, artes mistas
    # level_2 - Organico, figurativo, figura humanda, abstrato, geometrico

    return GbsData(
        resource=resource,
        resource_price=resource_price,
        product=product,
        product_parts=product_parts,
        product_part_resources=product_part_resources,
        composition=composition,
        composition_library=composition_library,
    )

# Etl


In [ ]:
teste = {
    "id": "8b3229d9-fd6b-c336-bb10-862780a1eb75",
    "name": "Calmaria_teste",
    "raw_description": "<p>Artista: Ananda Sant´Anna</p><p>Ano da obra:&nbsp; 2023</p><p>Dimensões totais: 58,5cm x 140cm x 5cm (L x A x P)&nbsp;</p><p>Peso: 10000g</p><p>Escultura com cordas de polipropileno e chapa recostada de aço corten.</p><p>Peça Única. <span><span>Pronta-entrega.</span></span></p><p>Código produto: SER000360</p><p>{Abstrato, geométrico, Santa Luzia}</p>",
    "image_urls": [f"{PathConsts.DATA_PATH}/images/monalisa.png"],
    "artist_name": "Ananda Sant´Anna",
    "collection_name": None,
    "dimensions": [
        {"name": None, "width": 58.5, "height": 140.0, "depth": 5.0, "weight": 10000.0}
    ],
    "colors_tags": [],
    "materials_tags": ["cordas de polipropileno", "chapa recostada de aço corten"],
    "code": "SER000360",
    "year": 2023,
    "delivery_information": "Pronta-entrega",
    "stock_information": "Peça Única",
    "presentation_text": None,
    "type": "escultura",
    "extra_description": "Estilo abstrato e geométrico. Santa Luzia.",
    "url": "https://www.aviwa.com.br/product-page/cópia-de-opostos",
    "price": 1400,
}

art = AvwArt.model_validate(teste)

In [ ]:
gbs_data = get_gbs_entity_base_date(art)

In [ ]:
page = await start_page()

In [ ]:
login_proc = UiLoginProcessor(page, get_mapping_path("login.json"))
await login_proc.login(
    {
        "email": GbsConsts.GBS_USER,
        "password": GbsConsts.GBS_PASSWORD,
    }
)

In [ ]:
await asyncio.sleep(5)

resources_proc, resource_id = await create_or_update(
    page, "resources.json", gbs_data.resource
)

await asyncio.sleep(5)

resource_prices_proc, resource_price_id = await create_or_update(
    page,
    "resources-prices.json",
    gbs_data.resource_price,
    match=subdict(gbs_data.resource_price, ["identifier"]),
    url_params={"resource_id": resource_id},
)

await asyncio.sleep(5)

product_proc, product_id = await create_or_update(
    page, "products.json", gbs_data.product, is_modal=True
)

await asyncio.sleep(5)

product_parts_proc, product_part_id = await create_or_update(
    page,
    "products-parts.json",
    gbs_data.product_parts,
    url_params={"product_id": product_id},
)

await asyncio.sleep(5)

product_part_resources_proc, product_part_resource_id = await create_or_update(
    page,
    "products-parts-resources.json",
    gbs_data.product_part_resources
    | {
        "resource_id": resource_id  # Aqui é um select de entidades de Recurso
    },
    match={"resource": resource_id},
    match_type="contains",
    url_params={"part_id": product_part_id},
)

await asyncio.sleep(5)

# colocar aquii
gbs_data.composition["skp_file"] = f"{PathConsts.DATA_PATH}/raw_skps/M3D000012.skp"
compositions_proc, composition_id = await create_or_update(
    page, "compositions.json", gbs_data.composition
)

await asyncio.sleep(5)

composition_libraries_proc, composition_library_id = await create_or_update(
    page,
    "compositions-libraries.json",
    gbs_data.composition_library,
    match=subdict(gbs_data.composition_library, ["library"]),
    match_type="contains",
    url_params={"composition_id": composition_id},
)

In [ ]:
print(
    resource_id,
    resource_price_id,
    product_id,
    product_part_id,
    product_part_resource_id,
    composition_id,
    composition_library_id,
)

In [ ]:
# await display_page(page)